# 01 – Il neurone artificiale e il perceptron

In questo notebook:
1. costruiamo un neurone a mano (McCulloch e Pitts, 1943);
2. lo usiamo per le porte logiche AND, OR, NOT, NAND;
3. vediamo perché XOR è impossibile con un solo neurone;
4. facciamo imparare i pesi al perceptron (Rosenblatt, 1958).

Esegui le celle una alla volta con **Shift + Invio**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. Un neurone a mano

Il neurone fa due cose:
- **somma** gli ingressi, ognuno moltiplicato per il suo **peso**;
- si **attiva** (risponde 1) solo se la somma supera la **soglia**.

In [ ]:
def neurone(ingressi, pesi, soglia):
    somma = np.dot(ingressi, pesi)
    return 1 if somma > soglia else 0

### Esempio: vado al cinema?

Tre ingressi (1 = sì, 0 = no): bel tempo, ci sono gli amici, il cinema è vicino.
Gli amici contano il doppio.

In [ ]:
pesi = [2, 4, 2]   # bel tempo, amici, vicino
soglia = 5

print(neurone([1, 1, 0], pesi, soglia))  # bel tempo + amici
print(neurone([1, 0, 1], pesi, soglia))  # bel tempo + vicino, ma niente amici

**Prova tu:** cambia i pesi o la soglia. Che cosa serve perché tu vada al cinema anche da solo?

## 2. Le porte logiche

Con pesi e soglia giusti, lo stesso neurone diventa una porta logica.

In [ ]:
coppie = [(0, 0), (0, 1), (1, 0), (1, 1)]

def tabella(nome, pesi, soglia):
    print(nome)
    for x in coppie:
        print('  ', x, '->', neurone(x, pesi, soglia))

tabella('AND', [1, 1], 1.5)
tabella('OR', [1, 1], 0.5)
tabella('NAND', [-2, -2], -3)

In [ ]:
# NOT ha un solo ingresso
for x in [0, 1]:
    print('NOT', x, '->', neurone([x], [-1], -0.5))

AND e OR hanno gli **stessi pesi**: cambia solo la soglia. La soglia è il vero decisore.

## 3. XOR: il problema

XOR vale 1 solo se gli ingressi sono **diversi**.

Cerchiamo pesi e soglia provando migliaia di combinazioni.

In [ ]:
xor_atteso = [0, 1, 1, 0]
valori = np.arange(-3, 3.01, 0.25)
trovati = 0

for w1 in valori:
    for w2 in valori:
        for s in valori:
            uscite = [neurone(x, [w1, w2], s) for x in coppie]
            if uscite == xor_atteso:
                trovati += 1

print('Combinazioni provate:', len(valori) ** 3)
print('Combinazioni che fanno XOR:', trovati)

Nessuna. Il motivo si vede a occhio: un neurone traccia **una retta**.

Per AND una retta che separa gli 1 (quadrati) dagli 0 (cerchi) esiste; per XOR no.

In [ ]:
# quadrato arancione = 1, cerchio blu = 0 (forme diverse: leggibile anche per chi è daltonico)
fig, (sx, dx) = plt.subplots(1, 2, figsize=(13, 5.5))
fig.suptitle('AND e XOR: perché per AND basta una retta e per XOR no', fontsize=14)

for ax, titolo, uscite in [(sx, 'AND\n(separabile con una retta)', [0, 0, 0, 1]),
                           (dx, 'XOR\n(NON separabile con una retta)', [0, 1, 1, 0])]:
    for (x1, x2), uscita in zip(coppie, uscite):
        if uscita == 1:
            ax.scatter(x1, x2, marker='s', s=350, color='#C25A00', edgecolor='black', linewidth=2, zorder=3)
        else:
            ax.scatter(x1, x2, marker='o', s=350, color='#1F3A5F', edgecolor='black', linewidth=2, zorder=3)
        ax.annotate(f'({x1},{x2})→{uscita}', (x1, x2), xytext=(12, -16),
                    textcoords='offset points', fontweight='bold')
    ax.set_xlim(-0.5, 1.5); ax.set_ylim(-0.5, 1.5)
    ax.set_xlabel('x1'); ax.set_ylabel('x2')
    ax.set_title(titolo); ax.grid(alpha=0.3); ax.set_aspect('equal')

xs = np.linspace(-0.5, 1.5, 100)

# AND: la retta x1 + x2 = 1,5 divide il piano in due zone
sx.fill_between(xs, -0.5, 1.5 - xs, color='#DCE6F2')
sx.fill_between(xs, 1.5 - xs, 1.5, facecolor='#FCE2C4', hatch='//', edgecolor='#C25A00', linewidth=0)
sx.plot(xs, 1.5 - xs, color='#1F3A5F', linewidth=3)
sx.text(0.2, 1.35, 'Retta: x1 + x2 = 1,5', color='#1F3A5F', fontweight='bold')

# XOR: qualunque retta proviamo, sbaglia almeno un punto
dx.plot(xs, 0.5 - xs, '--', color='gray', linewidth=2)
dx.plot(xs, 0.8 - 0.5 * xs, ':', color='gray', linewidth=2)
dx.plot(xs, 1.5 - xs, '-.', color='gray', linewidth=2)
dx.text(0.05, 1.35, 'Nessuna retta funziona!', color='#C25A00', fontweight='bold')

plt.show()

### La soluzione: più neuroni in fila

XOR = (x1 OR x2) AND (x1 NAND x2). Due strati di neuroni bastano.
È l'idea alla base delle reti neurali profonde.

In [ ]:
def xor(x):
    a = neurone(x, [1, 1], 0.5)      # OR
    b = neurone(x, [-2, -2], -3)     # NAND
    return neurone([a, b], [1, 1], 1.5)  # AND

for x in coppie:
    print(x, '->', xor(x))

## 4. Il perceptron: pesi imparati, non scritti

Finora i pesi li abbiamo scelti noi. Il perceptron li **impara dagli esempi**:

1. parte con pesi a zero;
2. fa una previsione;
3. se sbaglia, sposta un po' i pesi nella direzione giusta;
4. ripete.

Al posto della soglia usiamo il **bias** (una soglia con il segno cambiato).

La regola di correzione è una riga sola: `peso += passo × errore × ingresso`.

In [ ]:
def addestra(X, y, epoche=50, passo=0.1):
    w = [0.0, 0.0]   # un peso per x1 e uno per x2
    b = 0.0          # bias
    storia = []
    for epoca in range(1, epoche + 1):
        errori = 0
        for (x1, x2), atteso in zip(X, y):
            somma = x1 * w[0] + x2 * w[1] + b
            previsione = 1 if somma > 0 else 0
            errore = atteso - previsione
            w[0] += passo * errore * x1
            w[1] += passo * errore * x2
            b += passo * errore
            errori += abs(errore)
        storia.append(errori)
        print(f'Epoca {epoca:2d}: errori {errori}   w1 = {w[0]:+.1f}   w2 = {w[1]:+.1f}   bias = {b:+.1f}')
        if errori == 0:
            print('Nessun errore: addestramento finito.')
            break
    return w, b, storia

### Impariamo NAND

NAND è la porta “universale”: combinando solo NAND si costruisce qualsiasi circuito logico.

In [ ]:
y_nand = [1, 1, 1, 0]
w, b, storia = addestra(coppie, y_nand)

A ogni epoca il perceptron rivede i 4 esempi e corregge i pesi.
Gli errori scendono a zero: ha trovato da solo dei pesi che fanno NAND.

### Il confine di decisione

Il perceptron divide il piano in due zone con una retta: nella zona arancione tratteggiata risponde 1, in quella azzurra risponde 0.
Un buon confine mette tutti i quadrati (1) nella zona arancione e tutti i cerchi (0) in quella azzurra.

In [ ]:
def disegna(w, b, y, titolo, X=coppie, limiti=(-0.5, 1.5, -0.5, 1.5), assi=('x1', 'x2')):
    x_min, x_max, y_min, y_max = limiti
    # le due zone: arancione tratteggiata = il perceptron risponde 1, azzurra = risponde 0
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
    zona = w[0] * xx + w[1] * yy + b
    plt.contourf(xx, yy, zona, levels=[-1e9, 0, 1e9],
                 colors=['#DCE6F2', '#FCE2C4'], hatches=['', '//'])
    plt.contour(xx, yy, zona, levels=[0], colors='#1F3A5F', linewidths=3)

    # gli esempi: quadrato = 1, cerchio = 0
    pochi = len(X) <= 4
    for (x1, x2), uscita in zip(X, y):
        if uscita == 1:
            plt.scatter(x1, x2, marker='s', s=350 if pochi else 60, color='#C25A00', edgecolor='black', zorder=3)
        else:
            plt.scatter(x1, x2, marker='o', s=350 if pochi else 60, color='#1F3A5F', edgecolor='black', zorder=3)
        if pochi:
            plt.annotate(f'({x1},{x2})→{uscita}', (x1, x2), xytext=(12, -16),
                         textcoords='offset points', fontweight='bold')

    plt.xlim(x_min, x_max); plt.ylim(y_min, y_max)
    plt.xlabel(assi[0]); plt.ylabel(assi[1])
    plt.title(titolo); plt.show()

disegna(w, b, y_nand, 'NAND: la retta imparata separa i punti')

**Prova tu:** allena AND (`[0, 0, 0, 1]`) e OR (`[0, 1, 1, 1]`) e disegna il confine.

### E XOR?

In [ ]:
y_xor = [0, 1, 1, 0]
w, b, storia = addestra(coppie, y_xor, epoche=15)

Gli errori restano a 4 e a fine epoca i pesi tornano sempre uguali: il perceptron gira in tondo, perché qualunque retta lascia almeno un punto dalla parte sbagliata (rivedi il grafico AND / XOR più sopra).

## 5. Un esempio vero: i fiori Iris

Un dataset nato nella statistica, prima dei computer: nel 1936 Ronald Fisher pubblicò le misure di 150 iris
(raccolte dal botanico Edgar Anderson) per separare le specie con una retta calcolata a mano.
Più tardi il Machine Learning lo ha adottato come esempio classico.

3 specie, 50 fiori ciascuna, 4 misure per fiore (lunghezza e larghezza di sepalo e petalo).
Fisher la retta la calcolava con una formula; il perceptron la **impara dagli esempi**.

| Setosa | Versicolor | Virginica |
|:---:|:---:|:---:|
| <img src="https://upload.wikimedia.org/wikipedia/commons/5/56/Kosaciec_szczecinkowaty_Iris_setosa.jpg" width="200"> | <img src="https://upload.wikimedia.org/wikipedia/commons/4/41/Iris_versicolor_3.jpg" width="200"> | <img src="https://upload.wikimedia.org/wikipedia/commons/9/9f/Iris_virginica.jpg" width="200"> |

Usiamo solo le misure del **petalo**, così restiamo su un piano e possiamo disegnare.

In [ ]:
import pandas as pd
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
fiori = iris.frame.rename(columns={'petal length (cm)': 'lunghezza_petalo',
                                   'petal width (cm)': 'larghezza_petalo'})
fiori['specie'] = iris.target_names[iris.target]
fiori[['specie', 'lunghezza_petalo', 'larghezza_petalo']].sample(8, random_state=0)

In [ ]:
fiori.specie.value_counts()

### Setosa o versicolor?

Teniamo solo queste due specie: versicolor = 1 (quadrato), setosa = 0 (cerchio).
Usiamo la stessa funzione `addestra` di prima, ma con 100 fiori veri invece di 4 coppie di 0 e 1.

In [ ]:
def prepara(specie_0, specie_1):
    scelti = fiori[fiori.specie.isin([specie_0, specie_1])]
    X = list(zip(scelti.lunghezza_petalo, scelti.larghezza_petalo))
    y = [1 if specie == specie_1 else 0 for specie in scelti.specie]
    return X, y

X_fiori, y_fiori = prepara('setosa', 'versicolor')
print(len(X_fiori), 'fiori')
w, b, storia = addestra(X_fiori, y_fiori)

In [ ]:
disegna(w, b, y_fiori, 'Setosa (cerchio) e versicolor (quadrato)', X=X_fiori,
        limiti=(0.5, 5.5, -0.2, 2.0), assi=('lunghezza petalo (cm)', 'larghezza petalo (cm)'))

Bastano poche epoche: le due specie hanno petali molto diversi e una retta le separa.

### Un fiore nuovo

Il modello è addestrato: ora lo **usiamo** su un fiore che non ha mai visto.

In [ ]:
w_fiori, b_fiori = w, b   # teniamo da parte i pesi imparati

def classifica(lunghezza, larghezza):
    somma = lunghezza * w_fiori[0] + larghezza * w_fiori[1] + b_fiori
    return 'versicolor' if somma > 0 else 'setosa'

print('Petalo 1,5 × 0,3 cm →', classifica(1.5, 0.3))
print('Petalo 4,2 × 1,3 cm →', classifica(4.2, 1.3))

**Prova tu:** inventa altre misure. Dove cambia la risposta?

### E con una foto?

Il nostro perceptron conosce solo **due numeri**: lunghezza e larghezza del petalo.
Una foto invece sono centinaia di migliaia di pixel: il perceptron non sa nemmeno dove sia il petalo.

Per riconoscere un fiore da una foto usiamo un modello già addestrato da altri: **CLIP** (OpenAI, 2021).

> **Nota – che cos'è CLIP e come ha imparato**
>
> - CLIP è una **rete profonda** con milioni di parametri. Tecnicamente è fatta di due transformer
>   (uno per le immagini, uno per il testo), ma qui l'architettura non è il punto.
> - **Non** ha imparato con l'apprendimento supervisionato classico: nessuno gli ha dato foto
>   etichettate “setosa”, “versicolor”, “virginica”.
> - Ha imparato da **centinaia di milioni di foto con la loro didascalia** trovate su internet,
>   imparando ad abbinare ogni immagine al suo testo. È una forma di supervisione più debole:
>   le “etichette” sono le didascalie, scritte da persone per altri scopi.
> - Per questo può riconoscere specie su cui nessuno lo ha addestrato apposta:
>   gli diamo tre frasi e lui dice quale descrive meglio la foto (*zero-shot*).

> **Nota – perché non facciamo come con il perceptron**
>
> | | Il nostro perceptron | CLIP |
> |---|---|---|
> | Ingresso | 2 numeri (misure del petalo) | una foto |
> | Esempi | 100 fiori etichettati | centinaia di milioni di foto con didascalia |
> | Chi lo addestra | noi, in pochi secondi | altri, con migliaia di GPU |
> | Tipo di apprendimento | supervisionato classico | supervisione debole (immagine + testo) |
>
> L'apprendimento supervisionato classico sulle foto richiederebbe centinaia di foto di iris,
> ognuna etichettata con la specie, e l'addestramento di una rete: troppo per un laboratorio.

La prossima cella scarica il modello. L'avviso su `HF_TOKEN` si può ignorare:
serve solo a chi ha un account Hugging Face e vuole scaricare più velocemente.

In [ ]:
import os
os.environ['HF_HUB_DISABLE_XET'] = '1'   # evita il download che in Colab a volte si blocca
import requests, torch
from io import BytesIO
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

nome = 'openai/clip-vit-base-patch32'      # circa 600 MB, un minuto per scaricarlo
clip = CLIPModel.from_pretrained(nome, use_safetensors=True)   # scarica il modello una volta sola
processore = CLIPProcessor.from_pretrained(nome)

specie = ['Iris setosa', 'Iris versicolor', 'Iris virginica']
frasi = [f'a photo of the flower {s}' for s in specie]   # CLIP ha imparato soprattutto in inglese

In [ ]:
import os

# Wikimedia rifiuta le richieste senza un User-Agent descrittivo
INTESTAZIONI = {'User-Agent': 'FormazioneAI/1.0 (notebook didattico; Python requests)'}

def carica_foto(indirizzo):
    if os.path.exists(indirizzo):                       # file caricato in Colab
        return Image.open(indirizzo).convert('RGB')
    if not indirizzo.startswith('http'):
        print('Incolla tra le virgolette l’indirizzo di una foto (deve iniziare con http).')
        return None
    risposta = requests.get(indirizzo, headers=INTESTAZIONI, timeout=30)
    tipo = risposta.headers.get('Content-Type', '')
    if risposta.status_code != 200 or not tipo.startswith('image'):
        print(f'Il sito non ha restituito una foto (codice {risposta.status_code}, tipo {tipo}).')
        print('Prova un altro indirizzo, oppure carica la foto in Colab (icona cartella a sinistra) e scrivi il nome del file.')
        return None
    return Image.open(BytesIO(risposta.content)).convert('RGB')


def riconosci(url):
    foto = carica_foto(url)
    if foto is None:
        return

    ingressi = processore(text=frasi, images=foto, return_tensors='pt', padding=True)
    with torch.no_grad():
        probabilita = clip(**ingressi).logits_per_image.softmax(dim=1)[0]

    plt.imshow(foto); plt.axis('off'); plt.show()
    for nome_specie, p in sorted(zip(specie, probabilita.tolist()), key=lambda r: -r[1]):
        print(f'{p:6.1%}  {nome_specie}')

riconosci('https://upload.wikimedia.org/wikipedia/commons/9/9f/Iris_virginica.jpg')

**Prova tu:** cerca la foto di un iris su internet, tasto destro → *Copia indirizzo immagine*, e incollala qui.
In alternativa carica una foto in Colab (icona cartella a sinistra) e scrivi il nome del file, per esempio `riconosci('iris.jpg')`.

In [ ]:
riconosci('incolla qui l’indirizzo della foto')

### E se non è un iris?

Proviamo con delle margherite.

In [ ]:
riconosci('https://upload.wikimedia.org/wikipedia/commons/6/63/Five_daisies_%28Bellis_perennis%29.jpg')

> **Nota – una probabilità, non una certezza**
>
> CLIP risponde comunque, e le percentuali sommano sempre al 100%: deve scegliere tra le tre frasi
> che gli abbiamo dato, anche se nessuna è giusta. Non sa dire “non è un iris” se non glielo chiediamo.
>
> Anche sulle foto di iris può sbagliare: le tre specie si somigliano molto.
> Come gli LLM, un modello dà la risposta **più probabile**, non quella **vera**.

### Versicolor o virginica?

Ora le due specie più simili: virginica = 1 (quadrato), versicolor = 0 (cerchio).

In [ ]:
X_fiori, y_fiori = prepara('versicolor', 'virginica')
w, b, storia = addestra(X_fiori, y_fiori, epoche=20)

In [ ]:
disegna(w, b, y_fiori, 'Versicolor (cerchio) e virginica (quadrato)', X=X_fiori,
        limiti=(2.8, 7.2, 0.8, 2.7), assi=('lunghezza petalo (cm)', 'larghezza petalo (cm)'))

Gli errori non arrivano a zero: alcuni fiori delle due specie hanno petali quasi uguali
e nessuna retta li separa tutti. È lo stesso problema di XOR, ma con dati veri.

Il perceptron continua a correggersi senza mai fermarsi. Servono modelli che accettano
qualche errore e cercano la retta che sbaglia meno: è quello che faremo con la regressione nel notebook 02.

## Da ricordare

- Un neurone = somma pesata + soglia.
- **Imparare = cambiare i pesi.**
- Un solo neurone traccia una retta: per problemi più difficili servono più strati.
- Con dati veri il perceptron funziona se una retta separa le classi (setosa / versicolor), altrimenti non si ferma mai (versicolor / virginica).